# Vector Databases & FAISS Playground

This notebook helps you understand:
1. **Embeddings** — how text becomes numbers
2. **Similarity** — how we compare those numbers
3. **FAISS** — a fast vector database for similarity search
4. **LangChain + FAISS** — the easy way to build semantic search

Run each cell in order. You can change the example sentences and queries to experiment.

## Step 0: Setup

Load environment variables. If you want to use OpenAI embeddings later, put your keys in a `.env` file:

```
OPENAI_API_KEY=sk-...
HF_TOKEN=hf_...
```

For this notebook we mostly use the free local `all-MiniLM-L6-v2` model, so no API key is required.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# Optional: set API keys from environment
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN", "")

## Step 1: Install required packages

Uncomment and run the lines below if you haven't installed these yet.

In [ ]:
# !pip install langchain-huggingface ipywidgets faiss-cpu scikit-learn

## Step 2: Create an embedding model

An embedding model turns text into a list of numbers (a vector).

We use `all-MiniLM-L6-v2` because it is:
- Free
- Runs locally on your machine
- Produces 384-dimensional vectors

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# Load the local embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("Embedding model loaded!")

### Embed a single sentence

See what an embedding looks like. It is just a long list of numbers.

In [ ]:
sentence = "Hello AI Agents"
embed = embeddings.embed_query(sentence)

print(f"Sentence: {sentence}")
print(f"Embedding length: {len(embed)} dimensions")
print(f"First 5 numbers: {embed[:5]}")

## Step 3: Compare sentences with Cosine Similarity

Cosine similarity measures the **angle** between two vectors.
- `1.0`  → same direction (very similar meaning)
- `0.0`  → unrelated
- `-1.0` → opposite direction (rare for text embeddings)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Some example documents
documents = [
    "What is the capital of Germany?",
    "Who is the president of Germany?",
    "What is the population of Germany?"
]

# A query that is closer in meaning to the second document
my_query = "Narendra Modi is the prime minister of which country?"

# Embed everything
document_embeddings = embeddings.embed_documents(documents)
query_embedding = embeddings.embed_query(my_query)

print(f"Documents: {len(documents)}")
print(f"Each embedding has {len(document_embeddings[0])} dimensions")
print(f"Query embedding has {len(query_embedding)} dimensions")

In [ ]:
# Compute cosine similarity between each document and the query
scores = cosine_similarity(document_embeddings, [query_embedding])

print("Cosine similarities with the query:")
for doc, score in zip(documents, scores.flatten()):
    print(f"  {score:.4f} → {doc}")

### Try another query

Change the query below and see how the similarity scores change.

In [ ]:
my_query = "Leader of countries"
query_embedding = embeddings.embed_query(my_query)

scores = cosine_similarity(document_embeddings, [query_embedding])

print(f"Query: {my_query}")
for doc, score in zip(documents, scores.flatten()):
    print(f"  {score:.4f} → {doc}")

## Step 4: Compare with Euclidean (L2) Distance

Euclidean distance measures the **straight-line distance** between two vector points.
- Smaller value → more similar
- `0.0` → identical

In [ ]:
from sklearn.metrics.pairwise import euclidean_distances

distances = euclidean_distances(document_embeddings, [query_embedding])

print("Euclidean distances from the query:")
for doc, dist in zip(documents, distances.flatten()):
    print(f"  {dist:.4f} → {doc}")

## Step 5: FAISS — Native (low-level) similarity search

FAISS (Facebook AI Similarity Search) is a library for fast nearest-neighbor search.

In this section we build a FAISS index manually so you can see how it works under the hood.

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Sentences to index
sentences = [
    "RAG improves LLM answers",
    "Embeddings capture meaning",
    "FAISS is a fast vector database"
]

# Create embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")
vectors = model.encode(sentences)

print(f"Indexed {len(sentences)} sentences")
print(f"Vector shape: {vectors.shape}")  # (number of sentences, 384)

In [ ]:
# 1. Create a FAISS index for L2 (Euclidean) distance
dimension = vectors.shape[1]  # 384
index = faiss.IndexFlatL2(dimension)

# 2. Add vectors (FAISS requires float32)
index.add(vectors.astype("float32"))

print(f"Total vectors in index: {index.ntotal}")

In [ ]:
# 3. Search
query = "How do we make LLMs more accurate?"
query_vector = model.encode([query]).astype("float32")

k = 2  # top-2 nearest neighbors
distances, indices = index.search(query_vector, k)

print(f"Query: {query}\n")
print("Top matches:")
for dist, idx in zip(distances[0], indices[0]):
    print(f"  [{idx}] distance={dist:.4f} → {sentences[idx]}")

## Step 6: FAISS with LangChain (high-level)

LangChain wraps FAISS so you don't have to manage raw vectors manually.
You just pass text strings and an embedding model.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [ ]:
# Build a vector store directly from text strings
texts = [
    "Hello, my dog is cute",
    "AI Agents are powerful",
    "AI is the future"
]

vector_store = FAISS.from_texts(texts, embeddings)

print("Vector store created from texts")

In [ ]:
# Search for similar text
query = "Tell me about AI"
results = vector_store.similarity_search(query, k=2)

print(f"Query: {query}\n")
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content}")

### Manual FAISS construction (full control)

If you want to create the FAISS index yourself and wire it into LangChain, you can do it like this.

In [ ]:
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

# Create an empty FAISS index
index = faiss.IndexFlatL2(384)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

# Add texts one by one
vector_store.add_texts(["Machine learning is amazing", "Deep learning is powerful"])

print(f"Total vectors: {vector_store.index.ntotal}")

In [ ]:
# Inspect the mapping from FAISS index → document
faiss_index = 0
docstore_id = vector_store.index_to_docstore_id[faiss_index]
print(f"FAISS index {faiss_index} → docstore id {docstore_id}")
print(vector_store.docstore.search(docstore_id))

In [ ]:
results = vector_store.similarity_search("What is ML?", k=2)
for doc in results:
    print(doc.page_content)

## Step 7: Save and load a FAISS index

You can persist the vector store to disk and reload it later.

In [ ]:
# Save
vector_store.save_local("faiss_index_demo")
print("Saved to faiss_index_demo/")

In [ ]:
# Load
loaded_store = FAISS.load_local(
    "faiss_index_demo",
    embeddings,
    allow_dangerous_deserialization=True  # safe only for indexes you created
)

results = loaded_store.similarity_search("AI", k=2)
for doc in results:
    print(doc.page_content)

## 🧪 Playground Exercises

1. **Change the query** in Step 3 and Step 5. Try: `"capital city"`, `"government"`, `"population statistics"`. Which document scores highest each time?

2. **Add more sentences** to the FAISS index in Step 5. Search again and see if the top result changes.

3. **Compare metrics** — for the same query, print both cosine similarity and L2 distance. Do they agree on which document is most similar?

4. **Build a tiny Q&A system** — create a list of 10 facts about a topic you like, store them in FAISS, and ask questions.

5. **Try OpenAI embeddings** — replace `HuggingFaceEmbeddings` with `OpenAIEmbeddings(model="text-embedding-3-small")` and compare the results. (Requires an OpenAI API key.)